In [ ]:
# 1. Import dan direktori proyek
from pathlib import Path
from datetime import datetime, timezone
from dataclasses import asdict
import json
import shutil
import sys

import pandas as pd
import torch

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "multimodal_mil_model.py").is_file() and (p / "app_data_filtered.csv").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from multimodal_mil_model import ModelConfig, OptimizerConfig
from experiment_runner import file_sha256, write_json
from unimodal_experiments import load_reference_data, train_independent, test_independent


In [ ]:
# 2. Parameter pemenang multimodal; hanya konfigurasi, bukan bobot terlatih
EXPERIMENT_DIR = ROOT / "experiments" / "Experiment 1"
REFERENCE_DIR = EXPERIMENT_DIR / "runs" / "search_20261018_044822_847109_seed1"
selection = json.loads((REFERENCE_DIR / "best_model.json").read_text(encoding="utf-8"))
assert selection["selection_final"]
scenario_id = selection["winner"]["scenario_id"]
assert scenario_id == "S26_pretrained_frozen_adam_P2", "Pemenang referensi berubah."
reference_trial = REFERENCE_DIR / "trials" / scenario_id
reference_config = json.loads((reference_trial / "config.json").read_text(encoding="utf-8"))

MODEL_CONFIG = ModelConfig(**reference_config["model"])
OPTIMIZER_CONFIG = OptimizerConfig(**reference_config["optimizer"])
SETTINGS = {**reference_config["training"], "device": "cuda" if torch.cuda.is_available() else "cpu"}
assert SETTINGS["seed"] == 1 and SETTINGS["monitor"] == "val_f1"
assert SETTINGS["patience"] == 25 and SETTINGS["early_stopping_rule"] == "best_monitor"
assert SETTINGS["epochs"] == 100 and MODEL_CONFIG.num_tabular_features == 53
MODALITIES = ("image", "clinical")

# None: dua pelatihan baru. Isi path hanya untuk membaca trial selesai dalam run yang sama.
# Trial gagal/interupsi tidak dilanjutkan; buat run baru untuk mengulang.
RESUME_RUN_DIR = None
run_id = "independent_unimodal_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f") + "_seed1"
RUN_DIR = Path(RESUME_RUN_DIR).resolve() if RESUME_RUN_DIR else EXPERIMENT_DIR / "runs" / run_id
source_files = ["unimodal_experiments.py", "multimodal_mil_model.py", "training.py",
                "experiment_runner.py", "experiment_evaluation.py", "image_preprocessing.py", "mil_diagnostics.py"]
PROVENANCE = {
    "reference_directory": str(REFERENCE_DIR.resolve()), "reference_scenario": scenario_id,
    "reference_config_sha256": file_sha256(reference_trial / "config.json"),
    "reference_artifact_sha256": {name: file_sha256(REFERENCE_DIR / name) for name in [
        "split_manifest.csv", "tabular_preprocessing.joblib", "image_stats.json", "cohort.csv",
        "train_image_manifest.csv", "val_image_manifest.csv", "test_image_manifest.csv"]},
    "source_sha256": {name: file_sha256(ROOT / name) for name in source_files},
    "initialization": "fresh independent model; ImageNet weights only for image backbone; no reference model_state loaded",
    "design": "fixed winning multimodal hyperparameters, no unimodal hyperparameter search",
}
if not RESUME_RUN_DIR:
    RUN_DIR.mkdir(parents=True, exist_ok=False)
    (RUN_DIR / "source").mkdir()
    for name in source_files:
        shutil.copy2(ROOT / name, RUN_DIR / "source" / name)
    notebook = EXPERIMENT_DIR / "Experiment 1 (Independent MIL-Attention and MLP).ipynb"
    shutil.copy2(notebook, RUN_DIR / "source" / notebook.name)
    write_json(RUN_DIR / "config.json", {"model": MODEL_CONFIG.to_dict(),
        "optimizer": asdict(OPTIMIZER_CONFIG), "training": SETTINGS, "provenance": PROVENANCE})
    shutil.copy2(REFERENCE_DIR / "split_manifest.csv", RUN_DIR / "split_manifest.csv")
print("Run:", RUN_DIR)
print("Parameter referensi:", scenario_id)
print(json.dumps(SETTINGS, indent=2))


Run: d:\Kuliah\Magister\Semester 1\Topik Dalam Pengolahan Citra Digital\Multimodal MIL-Attention based Hierarchical Classification Regensburg Pediatric Dataset\experiments\Experiment 1\runs\independent_unimodal_20261018_095448_957957_seed1
Parameter referensi: S26_pretrained_frozen_adam_P2
{
  "seed": 1,
  "epochs": 100,
  "batch_size": 16,
  "num_workers": 0,
  "use_amp": true,
  "monitor": "val_f1",
  "patience": 25,
  "min_delta": 0.0,
  "threshold": 0.5,
  "early_stopping_rule": "best_monitor",
  "test_bootstrap_resamples": 1000,
  "device": "cuda",
  "save_plots": true,
  "verbose": true
}


In [ ]:
# 3. Pasien dan preprocessing identik; tidak split atau fit ulang
datasets, feature_cols = load_reference_data(ROOT, REFERENCE_DIR)
assert len(feature_cols) == 53 and "Length_of_Stay" in feature_cols
assert {name: len(datasets[name]) for name in ("train", "val", "test")} == {"train": 369, "val": 93, "test": 116}
print({name: len(value) for name, value in datasets.items()})
print("Fitur klinis:", len(feature_cols))
# MLP hanya membaca tabular/label/metadata jumlah citra, tanpa membuka berkas citra.
# MIL memakai preprocessing/augmentasi citra yang sama dengan referensi.


{'train': 369, 'train_eval': 369, 'val': 93, 'test': 116}
Fitur klinis: 53


In [ ]:
# 4. Dua pelatihan independen, dijalankan serial
# Image: ResNet18 frozen -> dual gated MIL-attention -> dua MLP tugas -> head.
# Clinical: MLP klinis -> dua MLP tugas -> head. Tidak ada backbone atau attention citra.
# Tidak ada concat/embedding nol. Lebar MLP tugas [256, 64] tetap sama.
results = {}
for modality in MODALITIES:
    print("Training:", modality)
    results[modality] = train_independent(
        RUN_DIR / modality, modality, MODEL_CONFIG, OPTIMIZER_CONFIG,
        datasets, SETTINGS, PROVENANCE,
    )


Training: image
Epoch 01/100 | lr 1.0e-03 | 34.9 dtk
  train | loss 1.2519 | acc 0.4918 | prec 0.5810 | sens 0.3359 | spec 0.6662 | F1 0.2348 | AUROC 0.4743
  val   | loss 1.1901 | acc 0.4946 | prec 0.1649 | sens 0.3333 | spec 0.6667 | F1 0.2206 | AUROC 0.5155  *terbaik*
Epoch 02/100 | lr 1.0e-03 | 8.8 dtk
  train | loss 1.2391 | acc 0.4918 | prec 0.1648 | sens 0.3315 | spec 0.6657 | F1 0.2202 | AUROC 0.5143
  val   | loss 1.2508 | acc 0.4946 | prec 0.1649 | sens 0.3333 | spec 0.6667 | F1 0.2206 | AUROC 0.5697
Epoch 03/100 | lr 1.0e-03 | 8.5 dtk
  train | loss 1.2150 | acc 0.4891 | prec 0.2746 | sens 0.3323 | spec 0.6657 | F1 0.2238 | AUROC 0.5256
  val   | loss 1.1858 | acc 0.4946 | prec 0.1649 | sens 0.3333 | spec 0.6667 | F1 0.2206 | AUROC 0.5681  *terbaik*
Epoch 04/100 | lr 1.0e-03 | 7.8 dtk
  train | loss 1.2098 | acc 0.4946 | prec 0.3315 | sens 0.3360 | spec 0.6689 | F1 0.2258 | AUROC 0.5410
  val   | loss 1.2138 | acc 0.4946 | prec 0.1649 | sens 0.3333 | spec 0.6667 | F1 0.2206 

ValueError: Restored checkpoint did not reproduce best macro-F1

In [ ]:
# 5. Perbandingan train/validation pada checkpoint terbaik masing-masing
# Kedua kontrol tetap dilaporkan; tidak memilih salah satu berdasarkan test.
reference_result = json.loads((reference_trial / "result.json").read_text(encoding="utf-8"))
rows = []
for modality, result in results.items():
    for split in ("train", "val"):
        metrics = result[split]
        rows.append({"model": modality, "split": split, "best_epoch": result["best_epoch"],
            "accuracy": metrics["overall"]["accuracy"], "macro_f1": metrics["overall"]["f1"],
            "macro_auroc": metrics["auroc"], "diagnosis_auroc": metrics["diagnosis_auroc"],
            "severity_direct_auroc": metrics["severity_auroc"]})
for split in ("train", "val"):
    metrics = json.loads((reference_trial / f"{split}_metrics.json").read_text(encoding="utf-8"))
    rows.append({"model": "multimodal_reference", "split": split,
        "best_epoch": reference_result["best_epoch"], "accuracy": metrics["overall"]["accuracy"],
        "macro_f1": metrics["overall"]["f1"], "macro_auroc": metrics["auroc"],
        "diagnosis_auroc": metrics["diagnosis_auroc"], "severity_direct_auroc": metrics["severity_auroc"]})
validation_comparison = pd.DataFrame(rows)
validation_comparison.to_csv(RUN_DIR / "train_val_comparison.csv", index=False)
print(validation_comparison.to_string(index=False))


In [ ]:
# 6. Testing dua kontrol setelah checkpoint dikunci berdasarkan validation
EVALUATE_TEST = False  # True setelah training selesai; jalankan ulang cell ini saja.

if EVALUATE_TEST:
    summaries = []
    for modality in MODALITIES:
        report = test_independent(RUN_DIR / modality, datasets["test"], SETTINGS["test_bootstrap_resamples"])
        directory = RUN_DIR / modality / "testing"
        for filename in ("overview.csv", "per_class.csv", "binary_tasks.csv", "bootstrap_95ci.csv"):
            print("\n", modality, filename)
            print(pd.read_csv(directory / filename).to_string(index=False))
        overview = pd.read_csv(directory / "overview.csv")
        overview["model"] = modality + "_independently_trained"
        summaries.append(overview)
        print("Laporan:", directory / "report.html")
    # Hasil multimodal referensi dibaca saja; tidak dievaluasi atau dilatih ulang.
    original = pd.read_csv(reference_trial / "testing" / "overview.csv")
    original = original[original["mode"].eq("multimodal")].copy()
    original["model"] = "multimodal_reference"
    summaries.append(original)
    comparison = pd.concat(summaries, ignore_index=True)
    comparison.to_csv(RUN_DIR / "test_comparison.csv", index=False)
    print(comparison[["model", "group", "patients", "accuracy", "macro_f1", "auroc_macro_ovr"]].to_string(index=False))
else:
    print("Testing belum dijalankan. Training/validation dua kontrol disimpan di:", RUN_DIR)
# Encoder/head masing-masing sudah tersimpan di best_model.pt untuk studi fusion berikutnya.
# Fusion dari encoder terlatih terpisah belum dijalankan pada notebook ini.
